# vamprhyme-llm on Colab

Runtime > Change runtime type > **T4 GPU** before running anything. See `colab.md` for what each cell does.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUNS = "/content/drive/MyDrive/vamprhyme/runs"
!mkdir -p {RUNS}

## 3. Get the project

In [ ]:
import os
from google.colab import userdata

REPO = "YOUR_GITHUB_USERNAME/vamprhyme-llm"
PRIVATE = True

if PRIVATE:
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
else:
    url = f"https://github.com/{REPO}.git"

if os.path.isdir("/content/vamprhyme-llm"):
    !git -C /content/vamprhyme-llm pull
else:
    !git clone -q {url} /content/vamprhyme-llm

%cd /content/vamprhyme-llm

## 3b. Copy the token data from Drive

In [ ]:
!unzip -oq /content/drive/MyDrive/vamprhyme/vamprhyme-tokens.zip -d /content/vamprhyme-llm
!ls -la data/tokens

## 4. Speed check: the Step 8 model on the GPU

In [ ]:
!python train_gpt.py --styles rhymes horror bridge --style-weights 1 1 1 \
    --steps 300 --eval-every 100 --eval-iters 20 \
    --device cuda --out /content/runs/gpu-check

## 5. Train the larger model

In [ ]:
!python train_gpt.py --styles rhymes horror bridge --style-weights 1 1 1 \
    --n-embd 256 --n-layer 6 --n-head 8 --block-size 256 --dropout 0.2 \
    --steps 5000 --eval-iters 50 \
    --device cuda --out {RUNS}/gpt-mix-big

## 6. Generate

In [ ]:
CKPT = f"{RUNS}/gpt-mix-big/model.pt"

!python sample.py --checkpoint {CKPT} --device cuda --prompt "<|rhymes|>" --temperature 0.8
!python sample.py --checkpoint {CKPT} --device cuda --prompt "<|horror|>" --temperature 0.8
!python sample.py --checkpoint {CKPT} --device cuda --prompt "<|rhymes|>Twinkle, twinkle, little " \
    --blend-with horror --blend-weight 0.6 --temperature 0.8

## 7. Loss curves

In [ ]:
from IPython.display import Image
Image(f"{RUNS}/gpt-mix-big/loss.png")